# 02 — Nettoyage : une ligne par vente de logement

Objectif : passer des lignes brutes de DVF à un tableau propre où **chaque ligne est la vente
d'un seul logement** (maison ou appartement), avec son prix, ses caractéristiques et sa position.

Le résultat est sauvegardé dans `data/processed/logements_paca.parquet`.

In [ ]:
import pandas as pd
from pathlib import Path

# Remonter les dossiers jusqu'à trouver data/raw (fonctionne où que soit le notebook)
racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

fichiers = sorted((racine / "data" / "raw").glob("*.csv.gz"))
print(f"{len(fichiers)} fichiers trouvés dans {racine / 'data' / 'raw'}")

# Les codes (postal, commune, département, parcelle) sont des identifiants, pas des nombres :
# on les lit comme du texte pour garder les zéros de tête ("04" et non 4)
types_texte = {"code_postal": str, "code_commune": str, "code_departement": str, "id_parcelle": str}
df = pd.concat(
    (pd.read_csv(f, low_memory=False, dtype=types_texte) for f in fichiers),
    ignore_index=True,
)
print(f"Lignes brutes : {len(df):,}")

entonnoir = {"Lignes brutes": len(df)}

## 1. Ne garder que les ventes classiques

In [ ]:
# On écarte les échanges, adjudications, ventes en l'état futur d'achèvement (VEFA), etc.
ventes = df[df["nature_mutation"] == "Vente"].copy()
entonnoir["Lignes de type « Vente »"] = len(ventes)
entonnoir["Ventes uniques"] = ventes["id_mutation"].nunique()
print(f"Lignes 'Vente' : {len(ventes):,}")
print(f"Ventes uniques : {ventes['id_mutation'].nunique():,}")

## 2. Identifier les biens distincts de chaque vente (piège n°2)

In [ ]:
# Les lignes sans type_local sont du terrain nu : on les met de côté pour l'instant.
# Un même bien peut être répété (une ligne par nature de culture du terrain) : on dédoublonne.
locaux = ventes.dropna(subset=["type_local"]).drop_duplicates(
    subset=["id_mutation", "id_parcelle", "type_local",
            "surface_reelle_bati", "nombre_pieces_principales", "lot1_numero"]
)

# Pour chaque vente : combien de maisons, d'appartements, de dépendances, d'autres locaux ?
comptes = locaux.groupby("id_mutation")["type_local"].value_counts().unstack(fill_value=0)
comptes.head()

## 3. Garder les ventes d'un seul logement (piège n°1)

In [ ]:
nb_logements = comptes.get("Maison", 0) + comptes.get("Appartement", 0)
nb_dependances = comptes.get("Dépendance", 0)
nb_autres = comptes.sum(axis=1) - nb_logements - nb_dependances  # locaux commerciaux, etc.

ventes_simples = comptes.index[(nb_logements == 1) & (nb_autres == 0)]
entonnoir["Ventes avec au moins un bâtiment"] = len(comptes)
entonnoir["Ventes d'un seul logement"] = len(ventes_simples)
print(f"Ventes d'un seul logement : {len(ventes_simples):,} "
      f"sur {len(comptes):,} ventes avec au moins un bâtiment")

## 4. Construire le tableau final : une ligne par vente

In [ ]:
colonnes = ["id_mutation", "date_mutation", "valeur_fonciere", "type_local",
            "surface_reelle_bati", "nombre_pieces_principales",
            "code_postal", "code_commune", "nom_commune", "code_departement",
            "longitude", "latitude"]

logements = locaux[
    locaux["type_local"].isin(["Maison", "Appartement"])
    & locaux["id_mutation"].isin(ventes_simples)
][colonnes].set_index("id_mutation")

# Nombre de dépendances (caves, garages...) : ce sera une variable du modèle
logements["nb_dependances"] = nb_dependances.reindex(logements.index).fillna(0).astype(int)

# Surface de terrain totale de la vente, sans compter deux fois la même parcelle
terrain = (
    ventes.drop_duplicates(subset=["id_mutation", "id_parcelle", "code_nature_culture", "surface_terrain"])
    .groupby("id_mutation")["surface_terrain"].sum()
)
logements["surface_terrain"] = terrain.reindex(logements.index).fillna(0)
# Pour un appartement, le terrain est celui de toute la copropriété : sans intérêt
logements.loc[logements["type_local"] == "Appartement", "surface_terrain"] = 0

logements["date_mutation"] = pd.to_datetime(logements["date_mutation"])

print(f"Une ligne par vente ? {logements.index.is_unique}")
print(f"Logements : {len(logements):,}")
logements.head()

## 5. Supprimer les valeurs absurdes

In [ ]:
avant = len(logements)

logements = logements.dropna(subset=["valeur_fonciere", "surface_reelle_bati", "longitude", "latitude"])
logements = logements[
    (logements["valeur_fonciere"] >= 10_000)          # ventes symboliques
    & (logements["surface_reelle_bati"] >= 9)         # erreurs de saisie
    & (logements["nombre_pieces_principales"] >= 1)
]

# Prix au m² : on retire les 1 % les plus extrêmes de chaque côté, par type de bien
logements["prix_m2"] = logements["valeur_fonciere"] / logements["surface_reelle_bati"]
bas = logements.groupby("type_local")["prix_m2"].transform(lambda s: s.quantile(0.01))
haut = logements.groupby("type_local")["prix_m2"].transform(lambda s: s.quantile(0.99))
logements = logements[(logements["prix_m2"] >= bas) & (logements["prix_m2"] <= haut)]

entonnoir["Après suppression des valeurs aberrantes"] = len(logements)
print(f"Supprimés : {avant - len(logements):,} ({(avant - len(logements)) / avant:.1%})")
print(f"Restants : {len(logements):,}")
logements.groupby("type_local")["prix_m2"].describe()

> ✍️ **À toi** — explique pourquoi on a retiré ces ventes, et cite la limite de ce filtre (les 1 % les plus chers incluent de vraies ventes de luxe sur la Côte d'Azur).
>
> *(remplace ce bloc par ton commentaire)*

## 6. Résumé : l'entonnoir de nettoyage

In [ ]:
pd.DataFrame({"Nombre": entonnoir}).style.format("{:,.0f}")

## 7. Sauvegarde

In [ ]:
# Vérification : les codes doivent garder leurs zéros de tête ("04", "04800")
assert logements["code_departement"].str.len().eq(2).all(), "Codes département mal lus"

dossier_sortie = racine / "data" / "processed"
dossier_sortie.mkdir(exist_ok=True)
logements.to_parquet(dossier_sortie / "logements_paca.parquet")
print(f"Sauvegardé : {dossier_sortie / 'logements_paca.parquet'}")